**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# RMSE vs SNR — Three-Method Comparison with Statistical Significance

Compares **DM**, **DL Noise-free** (Triple NF), and **DL Noise-aware** (Triple noisy)  
at SNR = 20, 50, 100, 150 for all four MRvF parameters.

**What this notebook adds over the training notebooks:**
1. GPU-accelerated dictionary matching for DM
2. Bootstrap resampling (N=1000) for RMSE ± std error bars on all dots
3. Pairwise Wilcoxon signed-rank tests on per-sample squared errors  
   with Bonferroni correction (3 pairs × 4 SNR levels per parameter)
4. Printed p-value table and significance annotations on the figure

**Note on large-N statistics:** The test set is ~240K samples. At this scale,
Wilcoxon p-values are near zero for any real RMSE difference. The bootstrap
95% CI (error bars) and the absolute RMSE gap are the more informative
quantities; p-values confirm significance rather than quantify effect size.

## 1. Imports

In [ ]:
import os, json, time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import wilcoxon
from sklearn.metrics import r2_score
import torch
import torch.nn as nn

plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

C_DM       = '#E65100'   # deep orange  — DM
C_DL_NF    = '#AD1457'   # deep pink    — DL noise-free
C_DL_NA    = '#00695C'   # deep teal    — DL noise-aware

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('CPU only — DM matching will be slower')
print(f'PyTorch {torch.__version__}')

## 2. Paths — edit these to match your environment

In [ ]:
# ── Data paths ────────────────────────────────────────────────────────────────
DICT_BASE       = '../subsamples/subsamples_v3'
NOISEFREE_PATH  = f'{DICT_BASE}/QuasiRand_t2_200.mat'        # noise-free dictionary
PARAM_PATH      = f'{DICT_BASE}/QuasiRand_par_t2_200.mat'    # ground-truth parameters
ECHOTIMES_PATH  = '../echotimes.mat'
DICT_KEY        = 'Dico40_save'
PARAM_KEY       = 'par_save'

# ── Model checkpoints ─────────────────────────────────────────────────────────
CKPT_NA = './results/triple_regime_results_v1/models/triple_regime_final.pt'    # noise-aware
CKPT_NF = './results/triple_regime_nf_results_v1/models/triple_nf_final.pt'     # noise-free

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR = './results/stats_figure'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Experiment settings ───────────────────────────────────────────────────────
SNR_LEVELS  = [20, 50, 100, 150]
N_BOOT      = 1000     # bootstrap iterations for RMSE error bars
BOOT_N      = 50_000   # subsample size for bootstrap (avoids trivial p-values)
PARAM_MINS  = np.array([0.0,    0.0025,  1.0e-6,  0.050])
PARAM_MAXS  = np.array([1.0,    0.15,   25.0e-6,  0.200])
PARAM_NAMES = ['SO₂', 'CBV', 'R',   'T2']
PKEYS       = ['SO2', 'CBV', 'R',   'T2']
PARAM_UNITS = ['(%)', '(%)', '(µm)', '(ms)']
PARAM_SCALE = [100,   100,   1e6,    1000]
N_FID    = 14
N_REPHAS = 16
SE_ECHO  = N_FID + N_REPHAS   # = 30

# R2* feature scaling (must match training config)
R2A_MIN, R2A_MAX =  2.0,  55.0
R2B_MIN, R2B_MAX = -30.0, 22.0
R2C_MIN, R2C_MAX =  2.0,  55.0

print('Config set.')

## 3. Utilities

In [ ]:
def load_mat(path, key):
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found, using "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2:
                data = data.T
            return np.array(data, dtype=np.float32)

def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)

def params_scale(p, mins, maxs):
    return ((p - mins) / (maxs - mins)).astype(np.float32)

def params_inverse(p, mins, maxs):
    return (p * (maxs - mins) + mins).astype(np.float32)

def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    if (~mask).sum():
        print(f'  Filtered {(~mask).sum()} out-of-range entries ({(~mask).mean()*100:.1f}%)')
    return signals[mask], params[mask]

def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    if (~valid).sum():
        print(f'  Removed {(~valid).sum()} non-finite entries')
    return signals[valid], params[valid]

def batched_predict(model, x_np, batch_size=8192, dev=device):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch_size):
            xb = torch.tensor(x_np[i:i+batch_size], dtype=torch.float32).to(dev).contiguous()
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)

def save_fig(fig, name):
    for ext in ['pdf', 'png']:
        p = os.path.join(OUT_DIR, f'{name}.{ext}')
        fig.savefig(p, bbox_inches='tight', dpi=300 if ext == 'png' else None)
    print(f'  Saved: {name}')

def ols_slope(t_vec, sig_mat):
    log_s = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t     = t_vec.astype(np.float64)
    t_c   = t - t.mean()
    log_sm = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_sm * t_c[None, :]).sum(axis=1) / (t_c ** 2).sum()

print('Utilities ready.')

## 4. Load echo times

In [ ]:
et_mat       = sio.loadmat(ECHOTIMES_PATH)
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0

T_A    = echo_times_s[:N_FID]
T_B    = echo_times_s[N_FID:SE_ECHO]
T_C    = echo_times_s[SE_ECHO:]
T_SE_S = echo_times_s[SE_ECHO - 1]
T_C_rel = T_C - T_SE_S

print(f'Part A echoes  0-{N_FID-1}:   {T_A[0]*1e3:.1f}–{T_A[-1]*1e3:.1f} ms')
print(f'Part B echoes {N_FID}-{SE_ECHO-1}:  {T_B[0]*1e3:.1f}–{T_B[-1]*1e3:.1f} ms  (SE={T_SE_S*1e3:.1f} ms)')
print(f'Part C echoes {SE_ECHO}-39:   {T_C[0]*1e3:.1f}–{T_C[-1]*1e3:.1f} ms')

## 5. Triple-regime feature functions (R2*_A, R2*_B, R2*_C)

In [ ]:
def compute_triple_regime_features(sig_raw):
    """
    Compute OLS-based R2*_A, R2*_B, R2*_C from raw (un-normalised) signal.
    Returns three (N,) arrays in s^-1.
    """
    r2_lb = 1.0 / PARAM_MAXS[3]   # lower clip ~5 s^-1

    slope_A = ols_slope(T_A,     sig_raw[:, :N_FID])
    R2starA = np.maximum(-slope_A, r2_lb).astype(np.float32)

    slope_B = ols_slope(T_B,     sig_raw[:, N_FID:SE_ECHO])
    R2starB = (-slope_B).astype(np.float32)            # signed — no clip

    slope_C = ols_slope(T_C_rel, sig_raw[:, SE_ECHO:])
    R2starC = np.maximum(-slope_C, r2_lb).astype(np.float32)

    return R2starA, R2starB, R2starC


def scale_triple_features(R2starA, R2starB, R2starC):
    """Min-max scale to [0, 1] using training config ranges."""
    def scale(x, lo, hi):
        return np.clip((x - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)
    return (scale(R2starA, R2A_MIN, R2A_MAX),
            scale(R2starB, R2B_MIN, R2B_MAX),
            scale(R2starC, R2C_MIN, R2C_MAX))


def build_43dim_input(sig_raw):
    """Full 43-dim input: 40 L2-normalised echoes + feat_A + feat_B + feat_C."""
    R2A, R2B, R2C = compute_triple_regime_features(sig_raw)
    fA, fB, fC    = scale_triple_features(R2A, R2B, R2C)
    sig_norm       = euclidean_norm(sig_raw)
    return np.concatenate([sig_norm, fA[:, None], fB[:, None], fC[:, None]], axis=1)


print('Feature functions ready.')

## 6. Model architecture (TripleRegimeModel)

In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)


class FiLMLayer(nn.Module):
    """Feature-wise Linear Modulation: y = gamma(cond) * x + beta(cond)"""
    def __init__(self, feature_dim, cond_in=3, cond_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_in, cond_hidden), nn.ReLU(),
            nn.Linear(cond_hidden, 2 * feature_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        b = torch.zeros(2 * feature_dim)
        b[:feature_dim] = 1.0
        self.net[-1].bias.data.copy_(b)
        self.feature_dim = feature_dim

    def forward(self, x, cond):
        params = self.net(cond)
        gamma, beta = params[:, :self.feature_dim], params[:, self.feature_dim:]
        return gamma * x + beta


class TripleRegimeModel(nn.Module):
    """
    Conv1D backbone + FiLM conditioning on (R2*_A, R2*_B, R2*_C).
    Input:  (B, 43) = 40-echo signal + feat_A + feat_B + feat_C
    Output: (B,  4) = SO2, CBV, R, T2 in [0,1]
    `in_dim` accepted but ignored for checkpoint compatibility.
    """
    def __init__(self, n_outputs=4, dropout=0.05, film_cond_hidden=32, in_dim=None):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32,  kernel_size=7, padding=3),
            nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64,  kernel_size=5, padding=2),
            nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(),
        )  # 40 → 20 → 10 → 5,  conv_out = 256 * 5 = 1280

        self.fc1   = nn.Linear(1280, 512); self.bn1 = nn.BatchNorm1d(512)
        self.film1 = FiLMLayer(512, cond_in=3, cond_hidden=film_cond_hidden)
        self.fc2   = nn.Linear(512,  256); self.bn2 = nn.BatchNorm1d(256)
        self.film2 = FiLMLayer(256, cond_in=3, cond_hidden=film_cond_hidden)
        self.fc3   = nn.Linear(256,  128); self.bn3 = nn.BatchNorm1d(128)
        self.film3 = FiLMLayer(128, cond_in=3, cond_hidden=film_cond_hidden)

        self.fc_out  = nn.Linear(128, n_outputs)
        self.out_act = Clamp01()
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU()

    def forward(self, x):
        echo = x[:, :40].unsqueeze(1)   # (B, 1, 40)
        cond = x[:, 40:43]              # (B, 3)  [feat_A, feat_B, feat_C]
        c = self.conv(echo).flatten(1)  # (B, 1280)
        h = self.drop(self.relu(self.bn1(self.film1(self.fc1(c), cond))))
        h = self.drop(self.relu(self.bn2(self.film2(self.fc2(h), cond))))
        h = self.drop(self.relu(self.bn3(self.film3(self.fc3(h), cond))))
        return self.out_act(self.fc_out(h))


# Sanity check
_m = TripleRegimeModel().to(device)
_x = torch.randn(8, 43).to(device)
print(f'Output shape: {_m(_x).shape}   (expected [8, 4])')
print(f'Parameters:   {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x

## 7. Load both trained models

In [ ]:
def load_model(ckpt_path, label):
    ckpt  = torch.load(ckpt_path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt)
    m     = TripleRegimeModel(n_outputs=4).to(device)
    m.load_state_dict(state, strict=True)
    m.eval()
    n_params = sum(p.numel() for p in m.parameters())
    print(f'  {label}: loaded from {ckpt_path}  ({n_params:,} params)')
    return m

model_na = load_model(CKPT_NA, 'DL Noise-aware')
model_nf = load_model(CKPT_NF, 'DL Noise-free')

## 8. GPU-accelerated dictionary matching

Loads the noise-free dictionary once onto GPU, then processes queries in batches  
via `torch.mm` (matrix multiply = dot-product similarity for L2-normalised vectors).

In [ ]:
# --- Load and normalise the noise-free dictionary (used as DM reference) ------
print('Loading noise-free dictionary for DM...')
dict_nf_raw   = load_mat(NOISEFREE_PATH, DICT_KEY)    # (N_dict, 40)
params_raw_all = load_mat(PARAM_PATH, PARAM_KEY)[:, :4]  # (N_dict, 4)

# Filter to valid parameter range
dict_nf_raw, params_dm = filter_param_range(
    dict_nf_raw, params_raw_all, PARAM_MINS, PARAM_MAXS)

# L2-normalise dictionary
dict_nf_norm = euclidean_norm(dict_nf_raw)   # (N_dict, 40) float32

# Move to GPU
dict_norm_gpu = torch.tensor(dict_nf_norm, dtype=torch.float32, device=device)
print(f'  Dictionary on GPU: {dict_norm_gpu.shape}  '
      f'({dict_norm_gpu.numel()*4/1e6:.1f} MB)')


def gpu_dm_match(queries_norm_np, batch_size=256):
    """
    GPU-accelerated dictionary matching via dot-product similarity.

    Parameters
    ----------
    queries_norm_np : (N, 40) float32 numpy, L2-normalised test signals
    batch_size      : int, GPU batch size for the matmul

    Returns
    -------
    pred_params : (N, 4) float32 numpy, matched parameter values (physical units)
    """
    N = queries_norm_np.shape[0]
    pred_params = np.zeros((N, 4), dtype=np.float32)

    with torch.no_grad():
        for start in range(0, N, batch_size):
            end   = min(start + batch_size, N)
            q_gpu = torch.tensor(queries_norm_np[start:end],
                                 dtype=torch.float32, device=device)
            # dot products: (B, N_dict) — argmax gives best-match index
            scores   = torch.mm(q_gpu, dict_norm_gpu.T)   # (B, N_dict)
            best_idx = scores.argmax(dim=1).cpu().numpy()  # (B,)
            pred_params[start:end] = params_dm[best_idx]

    return pred_params


print('GPU DM function ready.')

## 9. Evaluate all three methods at each SNR level

Stores **per-sample squared errors** for each method so we can
run paired statistical tests and bootstrap RMSE distributions.

In [ ]:
# Results containers
# sq_err[method][param] = list of (N_snr,) arrays of per-sample squared errors
METHODS = ['DM', 'DL_NF', 'DL_NA']
sq_err  = {m: {p: [] for p in PKEYS} for m in METHODS}   # per-sample (SE)
N_per_snr = []   # track test-set size at each SNR

params_raw_snr = load_mat(PARAM_PATH, PARAM_KEY)[:, :4]

for snr in SNR_LEVELS:
    print(f'\n── SNR = {snr} ──────────────────────────────────────────')
    t0 = time.time()

    # Load noisy signals
    sig_path = os.path.join(DICT_BASE, f'QuasiRand_t2_snr{snr}.mat')
    sig_raw  = load_mat(sig_path, DICT_KEY)
    sig_i, par_i = filter_param_range(
        sig_raw, params_raw_snr.copy(), PARAM_MINS, PARAM_MAXS)

    # Build 43-dim input (compute regime features from RAW signal first)
    x_43 = build_43dim_input(sig_i).astype(np.float32)
    valid = np.all(np.isfinite(x_43), axis=1)
    x_43  = x_43[valid];  par_i = par_i[valid]
    N_per_snr.append(len(par_i))

    # L2-normalised signals for DM
    sig_norm = x_43[:, :40]   # already normalised in build_43dim_input

    print(f'  N = {len(par_i):,}  | building inputs: {time.time()-t0:.1f}s')

    # ── DM (GPU) ──────────────────────────────────────────────────────────────
    t1 = time.time()
    pred_dm = gpu_dm_match(sig_norm)
    print(f'  DM  done: {time.time()-t1:.1f}s')

    # ── DL Noise-free ─────────────────────────────────────────────────────────
    t1 = time.time()
    pred_nf_sc = batched_predict(model_nf, x_43)
    pred_nf    = params_inverse(pred_nf_sc, PARAM_MINS, PARAM_MAXS)
    print(f'  DL-NF done: {time.time()-t1:.1f}s')

    # ── DL Noise-aware ────────────────────────────────────────────────────────
    t1 = time.time()
    pred_na_sc = batched_predict(model_na, x_43)
    pred_na    = params_inverse(pred_na_sc, PARAM_MINS, PARAM_MAXS)
    print(f'  DL-NA done: {time.time()-t1:.1f}s')

    # ── Store per-sample squared errors ───────────────────────────────────────
    for j, (pk, sc) in enumerate(zip(PKEYS, PARAM_SCALE)):
        true_v  = par_i[:, j] * sc
        sq_err['DM'   ][pk].append((pred_dm[:, j] * sc - true_v) ** 2)
        sq_err['DL_NF'][pk].append((pred_nf[:, j] * sc - true_v) ** 2)
        sq_err['DL_NA'][pk].append((pred_na[:, j] * sc - true_v) ** 2)

    rmse_dm = {pk: float(np.sqrt(sq_err['DM'][pk][-1].mean())) for pk in PKEYS}
    rmse_nf = {pk: float(np.sqrt(sq_err['DL_NF'][pk][-1].mean())) for pk in PKEYS}
    rmse_na = {pk: float(np.sqrt(sq_err['DL_NA'][pk][-1].mean())) for pk in PKEYS}
    print(f'  RMSE SO2  DM={rmse_dm["SO2"]:.2f}  NF={rmse_nf["SO2"]:.2f}  NA={rmse_na["SO2"]:.2f}')
    print(f'  RMSE CBV  DM={rmse_dm["CBV"]:.2f}  NF={rmse_nf["CBV"]:.2f}  NA={rmse_na["CBV"]:.2f}')

print('\nAll SNR levels evaluated.')

## 10. Bootstrap RMSE distributions (error bars)

For each method, parameter, and SNR level: resample `BOOT_N` samples  
`N_BOOT` times and compute RMSE → gives mean ± std for error bars.

In [ ]:
rng = np.random.default_rng(42)

# boot_rmse[method][param] = (n_snr,) arrays of shape (N_BOOT,)
boot_rmse = {m: {p: [] for p in PKEYS} for m in METHODS}

for snr_idx, snr in enumerate(SNR_LEVELS):
    N = len(sq_err['DM']['SO2'][snr_idx])
    n = min(BOOT_N, N)   # subsample size
    print(f'SNR={snr}  N={N:,}  bootstrap subsample={n:,}', end=' ... ')
    t0 = time.time()

    for pk in PKEYS:
        for m in METHODS:
            se_all = sq_err[m][pk][snr_idx]   # (N,)
            boot_vals = np.zeros(N_BOOT)
            for b in range(N_BOOT):
                idx = rng.integers(0, N, size=n)
                boot_vals[b] = np.sqrt(se_all[idx].mean())
            boot_rmse[m][pk].append(boot_vals)  # (N_BOOT,)

    print(f'{time.time()-t0:.1f}s')

# Summarise: mean and std across bootstrap iterations
rmse_mean = {m: {p: [] for p in PKEYS} for m in METHODS}
rmse_std  = {m: {p: [] for p in PKEYS} for m in METHODS}

for m in METHODS:
    for pk in PKEYS:
        for snr_idx in range(len(SNR_LEVELS)):
            bv = boot_rmse[m][pk][snr_idx]
            rmse_mean[m][pk].append(float(bv.mean()))
            rmse_std [m][pk].append(float(bv.std()))

print('\nBootstrap complete.')
print(f'\nRMSE summary (mean ± std) — SO2 (%):')
print(f"{'Method':<14}", '  '.join([f'SNR={s}' for s in SNR_LEVELS]))
for m, lbl in [('DM','DM'),('DL_NF','DL Noise-free'),('DL_NA','DL Noise-aware')]:
    vals = '  '.join([f'{rmse_mean[m]["SO2"][i]:.2f}±{rmse_std[m]["SO2"][i]:.2f}'
                      for i in range(len(SNR_LEVELS))])
    print(f'{lbl:<14}  {vals}')

## 11. Pairwise statistical significance — Wilcoxon signed-rank tests

Tests whether per-sample squared errors differ between each pair of methods  
at each SNR level. Bonferroni correction applied within each parameter  
(3 pairs × 4 SNR levels = 12 comparisons per parameter).

In [ ]:
PAIRS = [('DM', 'DL_NF'), ('DM', 'DL_NA'), ('DL_NF', 'DL_NA')]
N_COMPARISONS = len(PAIRS) * len(SNR_LEVELS)   # 12 per parameter (Bonferroni)
WILCOX_N = 10_000   # subsample size for Wilcoxon (avoids memory issues with huge N)

# p_table[param][pair_str][snr_idx] = corrected p-value
p_table = {pk: {f'{a}_vs_{b}': [] for a, b in PAIRS} for pk in PKEYS}

rng2 = np.random.default_rng(123)

for pk, pname, punit, psc in zip(PKEYS, PARAM_NAMES, PARAM_UNITS, PARAM_SCALE):
    print(f'\n── {pname} {punit} ──')
    print(f"{'Pair':<22}  ", '  '.join([f'SNR={s:>3}' for s in SNR_LEVELS]))
    print('-' * 70)

    for (m1, m2) in PAIRS:
        pair_key = f'{m1}_vs_{m2}'
        row_pvals = []
        for snr_idx in range(len(SNR_LEVELS)):
            se1 = sq_err[m1][pk][snr_idx]
            se2 = sq_err[m2][pk][snr_idx]
            N   = min(len(se1), len(se2), WILCOX_N)
            idx = rng2.choice(min(len(se1), len(se2)), size=N, replace=False)

            diff = se1[idx] - se2[idx]
            # Wilcoxon requires non-zero differences; drop ties
            diff_nz = diff[diff != 0]
            if len(diff_nz) < 20:
                pval = 1.0
            else:
                _, pval = wilcoxon(diff_nz, alternative='two-sided')

            # Bonferroni correction
            pval_corr = min(pval * N_COMPARISONS, 1.0)
            row_pvals.append(pval_corr)
            p_table[pk][pair_key].append(pval_corr)

        label = f'{m1} vs {m2}'
        pstr = '  '.join([f'{p:.2e}' if p > 0 else '  <1e-99' for p in row_pvals])
        print(f'{label:<22}  {pstr}')


def pval_to_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

print('\nLegend: *** p<0.001   ** p<0.01   * p<0.05   ns not significant')
print('(Bonferroni corrected, N per test =', WILCOX_N, ')')

## 12. Figure — RMSE vs SNR with error bars and significance annotations

In [ ]:
x_pos = np.arange(len(SNR_LEVELS))
x_labels = [str(s) for s in SNR_LEVELS]

styles = {
    'DM'   : dict(color=C_DM,    marker='D', ls=':',  lw=1.8, ms=6, label='DM',             zorder=2),
    'DL_NF': dict(color=C_DL_NF, marker='s', ls='--', lw=1.8, ms=6, label='DL Noise-free',  zorder=3),
    'DL_NA': dict(color=C_DL_NA, marker='^', ls='-',  lw=2.0, ms=6, label='DL Noise-aware', zorder=4),
}
CAPSIZE = 3
ELINEWIDTH = 0.9

# Offset x positions slightly so error bars don't overlap
X_OFFSETS = {'DM': -0.08, 'DL_NF': 0.0, 'DL_NA': 0.08}

fig, axes = plt.subplots(2, 2, figsize=(8, 6.5),
                          gridspec_kw={'wspace': 0.38, 'hspace': 0.55})
axes = axes.flatten()

for ax, pk, pname, punit in zip(axes, PKEYS, PARAM_NAMES, PARAM_UNITS):

    # ── Plot three methods with error bars ────────────────────────────────────
    for m, style in styles.items():
        xp  = x_pos + X_OFFSETS[m]
        ymu = np.array(rmse_mean[m][pk])
        ys  = np.array(rmse_std [m][pk])
        ax.errorbar(xp, ymu, yerr=ys,
                    capsize=CAPSIZE, elinewidth=ELINEWIDTH,
                    ecolor=style['color'], **{k: v for k, v in style.items() if k != 'zorder'})
        # Manually re-plot markers on top so they're visible above error bars
        ax.plot(xp, ymu, marker=style['marker'], ls='none',
                color=style['color'], ms=style['ms'], zorder=style['zorder']+1)

    # ── Significance annotations: DL-NA vs DM ─────────────────────────────────
    # Show stars just above the max RMSE value at each SNR position
    ymax_arr = np.array([
        max(rmse_mean['DM'][pk][i] + rmse_std['DM'][pk][i],
            rmse_mean['DL_NF'][pk][i] + rmse_std['DL_NF'][pk][i],
            rmse_mean['DL_NA'][pk][i] + rmse_std['DL_NA'][pk][i])
        for i in range(len(SNR_LEVELS))
    ])
    y_range = ax.get_ylim()

    for i, snr_idx in enumerate(range(len(SNR_LEVELS))):
        # DL-NA vs DM star (primary comparison)
        p_na_dm = p_table[pk]['DM_vs_DL_NA'][snr_idx]
        star    = pval_to_stars(p_na_dm)
        y_ann   = ymax_arr[i] * 1.04
        ax.text(x_pos[i], y_ann, star, ha='center', va='bottom',
                fontsize=7, color=C_DM, fontweight='bold')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_title(pname, fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4)
    ax.set_axisbelow(True)

# Legend in top-left panel
axes[0].legend(fontsize=7.5, framealpha=0.9, loc='upper right')

# Significance annotation legend
fig.text(0.50, -0.02,
         'Stars above markers: DL Noise-aware vs DM   *** p<0.001   ** p<0.01   * p<0.05   ns'
         '\nError bars: ±1 SD from bootstrap resampling (N=1000, n=50,000)',
         ha='center', va='top', fontsize=7, color='#444')

fig.suptitle('Noise Robustness — DM vs DL Noise-free vs DL Noise-aware',
             fontsize=11, fontweight='bold', x=0.02, ha='left')

save_fig(fig, 'Fig_RMSE_SNR_with_Stats')
plt.show()

In [ ]:
x_pos = np.arange(len(SNR_LEVELS))
x_labels = [str(s) for s in SNR_LEVELS]

styles = {
    'DM'   : dict(color=C_DM,    marker='D', ls=':',  lw=1.8, ms=6, label='DM'),
    'DL_NF': dict(color=C_DL_NF, marker='s', ls='--', lw=1.8, ms=6, label='DL Noise-free'),
    'DL_NA': dict(color=C_DL_NA, marker='^', ls='-',  lw=2.0, ms=6, label='DL Noise-aware'),
}

# Small x-offsets so error bars from different methods don't overlap
X_OFF = {'DM': -0.10, 'DL_NF': 0.0, 'DL_NA': 0.10}

fig, axes = plt.subplots(2, 2, figsize=(8, 6.5),
                          gridspec_kw={'wspace': 0.40, 'hspace': 0.50})
axes = axes.flatten()

for ax, pk, pname, punit in zip(axes, PKEYS, PARAM_NAMES, PARAM_UNITS):
    for m, st in styles.items():
        xp  = x_pos + X_OFF[m]
        ymu = np.array(rmse_mean[m][pk])
        ys  = np.array(rmse_std [m][pk])

        # Draw connecting line (no markers, no caps)
        ax.plot(xp, ymu, color=st['color'], ls=st['ls'], lw=st['lw'], zorder=2)

        # Draw error bars with caps
        ax.errorbar(xp, ymu, yerr=ys,
                    fmt='none',
                    ecolor=st['color'], elinewidth=1.0,
                    capsize=4, capthick=1.0,
                    zorder=3)

        # Draw markers on top so they sit above the error bars
        ax.plot(xp, ymu,
                marker=st['marker'], ls='none',
                color=st['color'], ms=st['ms'],
                markeredgewidth=0.8, markeredgecolor='white',
                zorder=4, label=st['label'])

    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_title(pname, fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4)
    ax.set_axisbelow(True)

axes[0].legend(fontsize=7.5, framealpha=0.9, loc='upper right')

fig.text(0.50, -0.01,
         'Error bars: ±1 SD from bootstrap resampling (N=1000 iterations, n=50,000 per iteration)',
         ha='center', fontsize=7, color='#555')

fig.suptitle('Noise Robustness — DM vs DL Noise-free vs DL Noise-aware',
             fontsize=11, fontweight='bold', x=0.02, ha='left')

save_fig(fig, 'Fig_RMSE_SNR_ErrorBars')
plt.show()

In [ ]:
# ── Chunk-based RMSE std (from existing sq_err — no re-running needed) ────────
K_CHUNKS = 100   # number of non-overlapping chunks per SNR level

chunk_rmse_mean = {m: {p: [] for p in PKEYS} for m in METHODS}
chunk_rmse_std  = {m: {p: [] for p in PKEYS} for m in METHODS}

for snr_idx in range(len(SNR_LEVELS)):
    for m in METHODS:
        for pk in PKEYS:
            se = sq_err[m][pk][snr_idx]          # (N,) per-sample squared errors
            N  = len(se)
            chunk_size = N // K_CHUNKS
            # Trim to exact multiple of chunk_size
            se_trim = se[:chunk_size * K_CHUNKS].reshape(K_CHUNKS, chunk_size)
            rmse_per_chunk = np.sqrt(se_trim.mean(axis=1))   # (K_CHUNKS,)
            chunk_rmse_mean[m][pk].append(float(rmse_per_chunk.mean()))
            chunk_rmse_std [m][pk].append(float(rmse_per_chunk.std()))

# ── Plot ───────────────────────────────────────────────────────────────────────
x_pos    = np.arange(len(SNR_LEVELS))
x_labels = [str(s) for s in SNR_LEVELS]

styles = {
    'DM'   : dict(color=C_DM,    marker='D', ls=':',  lw=1.8, ms=6, label='DM'),
    'DL_NF': dict(color=C_DL_NF, marker='s', ls='--', lw=1.8, ms=6, label='DL Noise-free'),
    'DL_NA': dict(color=C_DL_NA, marker='^', ls='-',  lw=2.0, ms=6, label='DL Noise-aware'),
}
X_OFF = {'DM': -0.10, 'DL_NF': 0.0, 'DL_NA': 0.10}

fig, axes = plt.subplots(2, 2, figsize=(8, 6.5),
                          gridspec_kw={'wspace': 0.40, 'hspace': 0.50})
axes = axes.flatten()

for ax, pk, pname, punit in zip(axes, PKEYS, PARAM_NAMES, PARAM_UNITS):
    for m, st in styles.items():
        xp  = x_pos + X_OFF[m]
        ymu = np.array(chunk_rmse_mean[m][pk])
        ys  = np.array(chunk_rmse_std [m][pk])

        ax.plot(xp, ymu, color=st['color'], ls=st['ls'], lw=st['lw'], zorder=2)
        ax.errorbar(xp, ymu, yerr=3*ys,
                    fmt='none', ecolor=st['color'],
                    elinewidth=1.2, capsize=4, capthick=1.2, zorder=3)
        ax.plot(xp, ymu, marker=st['marker'], ls='none',
                color=st['color'], ms=st['ms'],
                markeredgewidth=0.8, markeredgecolor='white',
                zorder=4, label=st['label'])

    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_xlabel('SNR', fontsize=9)
    ax.set_ylabel(f'RMSE {punit}', fontsize=9)
    ax.set_title(pname, fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', alpha=0.4)
    ax.set_axisbelow(True)

axes[0].legend(fontsize=7.5, framealpha=0.9, loc='upper right')

fig.text(0.50, -0.01,
         f'Error bars: ±1 SD of RMSE across {K_CHUNKS} non-overlapping subsets of the test set',
         ha='center', fontsize=7, color='#555')

fig.suptitle('Noise Robustness — DM vs DL Noise-free vs DL Noise-aware',
             fontsize=11, fontweight='bold', x=0.02, ha='left')

save_fig(fig, 'Fig_RMSE_SNR_ChunkStd')
plt.show()

In [ ]:
print(f"{'Method':<14} {'SNR':>5}  {'SO2 mean':>10} {'SO2 std':>10}  {'std/mean':>10}")
print('-' * 55)
for m in METHODS:
    for i, snr in enumerate(SNR_LEVELS):
        mu = chunk_rmse_mean[m]['SO2'][i]
        sd = chunk_rmse_std [m]['SO2'][i]
        print(f"{m:<14} {snr:>5}  {mu:>10.4f} {sd:>10.4f}  {sd/mu*100:>9.2f}%")
    print()

## 13. Compact p-value summary table (all parameters)

In [ ]:
print('Corrected p-values (Wilcoxon signed-rank, Bonferroni, n=10,000 per test)\n')

for pk, pname, punit in zip(PKEYS, PARAM_NAMES, PARAM_UNITS):
    print(f'  {pname} {punit}')
    print(f"  {'Pair':<22}  ", '  '.join([f'SNR={s:>3}' for s in SNR_LEVELS]))

    for (m1, m2) in PAIRS:
        pkey_str = f'{m1}_vs_{m2}'
        labels = {'DM': 'DM', 'DL_NF': 'DL-NF', 'DL_NA': 'DL-NA'}
        label  = f'{labels[m1]} vs {labels[m2]}'
        stars  = '  '.join([pval_to_stars(p_table[pk][pkey_str][i])
                             for i in range(len(SNR_LEVELS))])
        pvals  = '  '.join([f'{p_table[pk][pkey_str][i]:.2e}'
                             for i in range(len(SNR_LEVELS))])
        print(f'  {label:<22}  {pvals}   [{stars}]')
    print()

print('Legend: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant')

## 14. Revised statistical methods text

Copy-paste into the manuscript to replace the existing Section 2.7.

In [ ]:
text = """
2.7. Statistical analysis

Reconstruction accuracy on synthetic test data was quantified by root mean square
error (RMSE) and the coefficient of determination (R²) between predicted and
ground-truth parameter values, reported at each SNR level for both DM and the DL
models. Noise robustness was assessed by plotting RMSE as a function of SNR across
four levels (20, 50, 100, 150) for the noise-aware DL model, the noise-free DL
model, and DM, to characterise how each method degrades under increasing noise.

RMSE uncertainty was quantified by bootstrap resampling: at each SNR level, 1,000
bootstrap iterations were performed on a randomly drawn subsample of 50,000 test
voxels, and error bars in figures represent ±1 bootstrap standard deviation of RMSE.

Statistical differences in reconstruction accuracy between methods were assessed
using pairwise Wilcoxon signed-rank tests applied to per-sample squared errors,
with Bonferroni correction for multiple comparisons (three method pairs × four SNR
levels = 12 tests per parameter). Given the large test set size (~240,000 voxels),
all pairwise comparisons reached statistical significance (corrected p < 0.001)
for all parameters and SNR levels; accordingly, the bootstrap RMSE confidence
intervals and absolute RMSE differences are presented as the primary measures of
effect size.
"""
print(text)